In [1]:
import hashlib
import os
import platform
import uuid

# ========= 修改为自己学号 =========
STUDENT_ID = 20244080226
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(f"{uuid.getnode()}|{platform.node()}|{platform.machine()}|{platform.system()}".encode()).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[机器指纹] {MACHINE_FP}")
print(f"[学号] {STUDENT_ID}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] base
[机器指纹] 5621e860ab4ac17e
[学号] 20244080226
[RUN_ID] 20244080226-5621e860ab4ac17e


## 字符串的乐趣

In [2]:
def is_palindrome(s: str) -> bool:
    processed = s.replace(" ", "").lower()
    return processed == processed[::-1]

print(is_palindrome("level"))          
print(is_palindrome("T A C O cat"))    
print(is_palindrome("hello"))         


True
True
False


In [3]:
def is_abecedarian(s: str) -> bool:
    processed = s.replace(" ", "")
    for i in range(len(processed)-1):
        if processed[i] > processed[i+1]:
            return False
    return True

print(is_abecedarian("adder"))             
print(is_abecedarian("abcd efgh xyz"))    
print(is_abecedarian("dog"))              


True
True
False


In [4]:
def remove_vowels(s: str) -> str:
    vowels = {"a", "e", "i", "o", "u"}
    res = []
    for ch in s:
        if ch.lower() not in vowels:
            res.append(ch)
    return "".join(res)

print(remove_vowels('cat'))    
print(remove_vowels('audio'))  
print(remove_vowels("aeiou")) 


ct
d



## 列表的乐趣

In [7]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("输入必须是列表")
    return lst[::-1]

print(list_reverse([1,2,3]))  

[3, 2, 1]


In [8]:
def is_sorted(seq) -> bool:
    for i in range(1, len(seq)):
        if seq[i] < seq[i-1]:
            return False
    return True

print(is_sorted([1,2,3,4]))  
print(is_sorted((1,3,2,4)))  
print(is_sorted([5]))       

True
False
True


## 元组表示向量与矩阵

In [9]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("第一个参数必须是元组(向量)")
    if not isinstance(s, (int, float)):
        raise TypeError("标量s必须是整数或浮点数")
    return tuple(x * s for x in t)

print(vec_scalar_mult((1,2,3), 2)) 


(2, 4, 6)


In [10]:
def vec_inner_product(x, y) -> float:
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("输入向量必须是元组")
    if len(x) != len(y):
        raise ValueError("两个向量维度必须相同")
    total = 0.0
    for xi, yi in zip(x, y):
        total += xi * yi
    return total

print(vec_inner_product((1,2), (3,4))) 


11.0


In [11]:
def check_valid_mx(m) -> bool:
    # 外层每个必须是元组
    for row in m:
        if not isinstance(row, tuple):
            return False
    if len(m) == 0:
        return True
    row_len = len(m[0])
    for row in m:
        if len(row) != row_len:
            return False
        for elem in row:
            if not isinstance(elem, (int, float)):
                return False
    return True

mx1 = ((1,2),(3,4))
print(check_valid_mx(mx1)) 
mx2 = ((1,2), (3,"a"))
print(check_valid_mx(mx2)) 


True
False


In [12]:
def mx_vec_mult(matrix, vec):
    if not check_valid_mx(matrix):
        raise ValueError("输入不是合法矩阵")
    if not isinstance(vec, tuple):
        raise TypeError("向量必须是元组")
    # 矩阵列数 = 第一行长度
    if len(matrix) > 0 and len(matrix[0]) != len(vec):
        raise ValueError("矩阵列数与向量维度不匹配")
    result = []
    for row in matrix:
        result.append(vec_inner_product(row, vec))
    return tuple(result)


M = ((1,2), (3,4))
v = (5,6)
print(mx_vec_mult(M, v)) 


(17.0, 39.0)


## 面向对象的Vector类

In [13]:
class Vector:
    def __init__(self, dim, elements=None):
        # 校验维度：非负整数
        if not isinstance(dim, int):
            raise TypeError("维度必须是整数")
        if dim < 0:
            raise ValueError("维度不能为负数")
        self.dim = dim

        if elements is None:
            # 默认：全部为0
            self.elements = [0.0 for _ in range(dim)]
        else:
            # elements是列表或者元组
            if not isinstance(elements, (list, tuple)):
                raise TypeError("elements必须是列表或元组")
            if len(elements) != dim:
                raise ValueError(f"维度{dim}与元素数量{len(elements)}不匹配")
            # 转为浮点数列表保存
            self.elements = [float(x) for x in elements]

    # -------- 比较运算符辅助函数 --------
    def _check_same_dim(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector对象进行比较")
        if self.dim != other.dim:
            raise ValueError("两个向量维度不一致，无法比较")

    def __eq__(self, other):
        self._check_same_dim(other)
        return self.elements == other.elements

    def __lt__(self, other):
        self._check_same_dim(other)
        return self.elements < other.elements

    def __le__(self, other):
        self._check_same_dim(other)
        return self.elements <= other.elements

    def __gt__(self, other):
        self._check_same_dim(other)
        return self.elements > other.elements

    def __ge__(self, other):
        self._check_same_dim(other)
        return self.elements >= other.elements

    # dot内积方法
    def dot(self, other):
        self._check_same_dim(other)
        s = 0.0
        for a,b in zip(self.elements, other.elements):
            s += a*b
        return s

    # v * x ；x可以标量或者Vector
    def __mul__(self, other):
        if isinstance(other, (int, float)):
            # 标量右乘
            new_elems = [x*other for x in self.elements]
            return Vector(self.dim, new_elems)
        elif isinstance(other, Vector):
            self._check_same_dim(other)
            new_elems = [a*b for a,b in zip(self.elements, other.elements)]
            return Vector(self.dim, new_elems)
        else:
            raise TypeError("乘法仅支持标量(int/float)或Vector对象")

    # 支持 2 * v 标量左乘
    def __rmul__(self, scalar):
        return self * scalar

    # 方便打印调试
    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"


In [14]:
v1 = Vector(2, (2,4))
v2 = Vector(2, (2,5))
print(v1)
print(v1 < v2)    
print(v1.dot(v2)) 

v3 = 2 * v1
print(v3)

v4 = Vector(2,(2,3))
v5 = Vector(2,(4,5))
v6 = v4 * v5
print(v6) 

Vector(2, [2.0, 4.0])
True
24.0
Vector(2, [4.0, 8.0])
Vector(2, [8.0, 15.0])
